# Ministral-3-14B — Authority pilot, Stage 1 (baseline, 0 engagement)

Does an authority cue in the poster's identity outrank factual correctness?

Every pairing puts **one "Dr. Remy Ashford" post (verified badge) against one plain
"Remy Ashford" post** — never Dr-vs-Dr. The two stimulus sets are identical apart from the
profile name and badge: same 100 pie charts, same claim texts, no engagement shown.

| # | Post A | Post B | tests |
|---|---|---|---|
| 1 | Dr. + **incorrect** | plain + **correct** | authority vs correctness — the headline |
| 2 | Dr. + correct | plain + correct | pure authority (correctness held equal) |
| 3 | Dr. + incorrect | plain + incorrect | pure authority (both wrong) |
| 4 | Dr. + correct | plain + incorrect | both cues agree — sanity check |

Pairings 2 and 3 make 1 interpretable: without them, choosing the Dr. post cannot be
distinguished from a blanket preference for "Dr." regardless of what the post claims.

**Read the A-rate before the preference.** Several models here answer a fixed slot on 96–100% of
trials and score ~50% mechanically (Section 5.1). `analyse_profile_paired` prints the A-rate and
declines to endorse a preference when one slot dominates. It runs after inference, reads only the
saved JSON, and cannot influence any answer.

Run top to bottom; restart the kernel after the setup cell as usual.

*Caveat: "Dr." and the verified badge vary together, so a positive result identifies an authority
bundle, not which cue did the work.*

In [1]:
import sys, subprocess

# 1. Uninstall torchaudio
subprocess.run([sys.executable, "-m", "pip", "uninstall", "torchaudio", "-y"])

# 2. Install PyTorch with CUDA 12.4
subprocess.run([sys.executable, "-m", "pip", "install",
    "torch==2.6.0", "torchvision==0.21.0",
    "--index-url", "https://download.pytorch.org/whl/cu124",
    "--user", "-q"], check=True)

# 3. Install latest transformers and accelerate (allowing pip to pull compatible tokenizers naturally)
subprocess.run([sys.executable, "-m", "pip", "install",
    "git+https://github.com/huggingface/transformers",
    "accelerate",
    "--user", "-q"], check=True)

print("✅ Installation complete — restart the kernel now")



ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.16.0.dev0 requires tokenizers<=0.23.0,>=0.22.0, but you have tokenizers 0.23.1 which is incompatible.


✅ Done — restart the kernel now


Restart the kernel after running the setup cell above.

In [2]:
!nvidia-smi

Fri Aug 21 14:32:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.127.08             Driver Version: 550.127.08     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:AF:00.0 Off |                    0 |
| N/A   38C    P0            132W /  700W |   23900MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
import sys
sys.path.append("/home/jovyan")
from config_hf_token import HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN)

from pathlib import Path
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "mistralai/Ministral-3-14B-Instruct-2512-BF16"
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
).eval()
processor = AutoProcessor.from_pretrained(MODEL_ID, fix_mistral_regex=True)
device = next(model.parameters()).device

# see experiments/e1/ministral-3-14b/e1-ministral-3-14b.ipynb for why this is needed
model.generation_config.max_length = None

ROOT_DIR = Path().resolve().parents[2]
sys.path.insert(0, str(ROOT_DIR / "experiments/e1"))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/585 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie model.language_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [4]:
import torch
free, total = torch.cuda.mem_get_info(0)          # actual device-wide free memory
print(torch.cuda.get_device_name(0))
print(f"VRAM total     : {total/1e9:.1f} GB")
print(f"VRAM free      : {free/1e9:.1f} GB")       # across ALL processes
print(f"this process   : {torch.cuda.memory_reserved(0)/1e9:.1f} GB reserved")

NVIDIA H100 80GB HBM3
VRAM total     : 84.9 GB
VRAM free      : 30.7 GB
this process   : 28.6 GB reserved


In [5]:
from e1_utils.inference_mistral import run_inference_mistral

In [6]:
from e1_utils.sampling import build_paired_sample
from e1_utils.e1_optimized import LIKE_PROMPT_PAIR
from e1_utils.e1_profile import run_e1_profile_paired, analyse_profile_paired

EXPERIMENT_DIR = Path().resolve().parent          # experiments/e1_authority/
OUTPUT_DIR = Path().resolve() / "outputs"
SEED = 42
SAMPLE_SIZE = 100                                  # selected_images.json copied from experiments/e1/

DR_C    = ROOT_DIR / "benchmarking/correct/dr-remy-ashford"
DR_I    = ROOT_DIR / "benchmarking/incorrect/dr-remy-ashford"
PLAIN_C = ROOT_DIR / "benchmarking/correct/remy-ashford"
PLAIN_I = ROOT_DIR / "benchmarking/incorrect/remy-ashford"
for d in (DR_C, DR_I, PLAIN_C, PLAIN_I):
    assert d.is_dir(), f"missing stimulus dir: {d}"

all_images = build_paired_sample(PLAIN_C, PLAIN_I, SEED, SAMPLE_SIZE, EXPERIMENT_DIR)
selected_numbers = [n.replace("_correct", "") for n, _ in all_images if "_correct" in n]
print(f"{len(selected_numbers)} posts")

PAIRINGS = {
    "1_authority_vs_correctness":      ({"dir": DR_I, "variant": "incorrect", "label": "dr"},
                                        {"dir": PLAIN_C, "variant": "correct",   "label": "plain"}),
    "2_pure_authority_both_correct":   ({"dir": DR_C, "variant": "correct",   "label": "dr"},
                                        {"dir": PLAIN_C, "variant": "correct",   "label": "plain"}),
    "3_pure_authority_both_incorrect": ({"dir": DR_I, "variant": "incorrect", "label": "dr"},
                                        {"dir": PLAIN_I, "variant": "incorrect", "label": "plain"}),
    "4_both_cues_agree":               ({"dir": DR_C, "variant": "correct",   "label": "dr"},
                                        {"dir": PLAIN_I, "variant": "incorrect", "label": "plain"}),
}

📋 Loading existing selection from /home/jovyan/conformity-llms-facebook-posts/experiments/e1_authority/selected_images.json
✅ All selected numbers verified in both correct and incorrect folders.
Selected 100 pairs → 200 images total
100 posts


In [7]:
INFERENCE_FN = run_inference_mistral

## Run the four pairings

In [8]:
for name, (side_a, side_b) in PAIRINGS.items():
    print(f"\n{'='*70}\n{name}\n{'='*70}")
    run_e1_profile_paired(selected_numbers, side_a, side_b, model, processor, device,
                          OUTPUT_DIR, SEED, prompt=LIKE_PROMPT_PAIR,
                          output_filename=f"e1_results_authority_{name}.json",
                          inference_fn=INFERENCE_FN)


1_authority_vs_correctness
✅ 001 → dr/incorrect (answered A)
✅ 002 → dr/incorrect (answered A)
✅ 003 → dr/incorrect (answered A)
✅ 004 → dr/incorrect (answered B)
✅ 005 → dr/incorrect (answered A)
✅ 006 → dr/incorrect (answered B)
✅ 007 → dr/incorrect (answered A)
✅ 008 → dr/incorrect (answered A)
✅ 009 → dr/incorrect (answered A)
✅ 010 → dr/incorrect (answered B)
✅ 011 → plain/correct (answered A)
✅ 012 → plain/correct (answered A)
✅ 013 → dr/incorrect (answered A)
✅ 014 → plain/correct (answered A)
✅ 015 → dr/incorrect (answered A)
✅ 016 → plain/correct (answered A)
✅ 017 → dr/incorrect (answered A)
✅ 018 → dr/incorrect (answered A)
✅ 019 → dr/incorrect (answered A)
✅ 020 → plain/correct (answered A)
✅ 021 → dr/incorrect (answered A)
✅ 022 → dr/incorrect (answered A)
✅ 023 → dr/incorrect (answered A)
✅ 024 → dr/incorrect (answered A)
✅ 025 → dr/incorrect (answered A)
✅ 026 → plain/correct (answered A)
✅ 027 → plain/correct (answered A)
✅ 028 → plain/correct (answered A)
✅ 029 → dr/i

## Results — read the A-rate first

In [9]:
for name in PAIRINGS:
    analyse_profile_paired(OUTPUT_DIR, f"e1_results_authority_{name}.json")


=== e1_results_authority_1_authority_vs_correctness.json ===
  n=100 valid=100 invalid=0
  chose the 'dr' post :  65.0%   (it sat in slot A 56.0% of trials)
  chose the CORRECT post      :  35.0%
  answered 'A'                :  91.0%
  ⚠ POSITIONAL DEFAULT — the model answers one slot on 91.0% of trials.
    Any preference above is an artifact of where each post happened to land, not a choice.
    Do not report it as a profile or correctness effect.

=== e1_results_authority_2_pure_authority_both_correct.json ===
  n=100 valid=100 invalid=0
  chose the 'dr' post :  56.0%   (it sat in slot A 56.0% of trials)
  (both sides correct — correctness is held constant, so any preference is pure profile)
  answered 'A'                : 100.0%
  ⚠ POSITIONAL DEFAULT — the model answers one slot on 100.0% of trials.
    Any preference above is an artifact of where each post happened to land, not a choice.
    Do not report it as a profile or correctness effect.

=== e1_results_authority_3_pure_a